# 03 — Обучение и независимый тест на сетке

В каждой точке всей сетки заново обучаем стратегию, затем проверяем на независимых
тестовых траекториях. Сохраняются стоимость, SE и парные разности «стратегия − удержание» для анализа в 04.
Старый запуск не используется. Выполните ячейки сверху вниз.


In [ ]:
from pathlib import Path
import numpy as np
from tqdm.auto import tqdm
from config import Config
from experiments.configuration import prepare_experiment
from experiments.statistics import load_experiment
from experiments.training import run_experiment
from experiments.legacy_grid import run_grid
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODE = "grid"  # совместимость со старым форматом; настройки ниже выбирают новый эксперимент

## Параметры нового эксперимента

`M_train` траекторий используются для обучения регрессий; `M_test` других —
для тестирования. Ещё `M_VALIDATION` независимых траекторий выбирают действие
в нулевой момент **до теста**. Тестовое среднее не обрезается через max(0, V).

Внутри каждого H выборки общие для разных μ. Train, validation и test независимы.
`RUN_ID` фиксирован: повторный запуск продолжает вычисления. При изменении параметров
укажите новый RUN_ID.
`BATCH_SIZE` задаёт размер блока обработки целых траекторий; регрессии используют всю обучающую выборку.

По умолчанию `COLLECT_PAIRED=True`: в том же тестовом пакете считаются награда
стратегии R и разность R − (μT + B(T)). Повторно вычислять сигнатуры для разностей
не нужно. Даже при остановке в нуле генерируется тестовая выборка для награды удержания.
Оба набора моментов сохраняются атомарно в одном checkpoint и продолжаются вместе.
Для нового прогона укажите новый RUN_ID. Для старых прогонов без парных моментов
остаётся отдельный пересчёт в 04; старые завершённые узлы 03 не пересчитывает.


In [2]:
MODE = "fresh"
RUN_ID = "daily21_21_train20k_test20k_v1"
cfg = Config(
    T=1.0, n_fine=21, n_exercise=21,
    M_train=20_000, M_test=20_000,
    K=4, seed=20261018,
    mu_grid=tuple(np.round(np.arange(-4.0, 9.001, 0.01), 6)),
    H_grid=tuple(np.round(np.arange(0.06, 1, 0.02), 6)),
)
BATCH_SIZE = 2_000
M_VALIDATION = 10_000
COLLECT_PAIRED = True  # парные моменты собираются сразу при первом тесте
RUN_DIR = ROOT / "result_optimal_stopping" / "runs" / RUN_ID
print("Каталог нового эксперимента:", RUN_DIR)
print(f"Узлов: {len(cfg.H_grid)*len(cfg.mu_grid)}; train={cfg.M_train}; test={cfg.M_test}")

Каталог нового эксперимента: /Users/armkilikia/optimal-stopping-fbm/result_optimal_stopping/runs/daily21_21_train20k_test20k_v1
Узлов: 61147; train=20000; test=20000


## Запуск и продолжение

Для каждого узла сохраняются обученная стратегия и прогресс теста по блокам.
Готовые узлы не пересчитываются. Во время теста показывается число обработанных
траекторий; после завершения узла — среднее и SE стоимости.
Для продолжения используйте те же RUN_ID и параметры.


In [3]:
if MODE == "fresh":
    manifest = prepare_experiment(RUN_DIR, cfg, batch_size=BATCH_SIZE, M_validation=M_VALIDATION)
    _, saved, _ = load_experiment(RUN_DIR, epsilon=None)
    with tqdm(total=manifest["family_size"], initial=int(saved.complete.sum()), desc="Узлы") as bar:
        def report(event):
            if event["stage"] == "node":
                bar.update(1)
                bar.set_postfix(H=event["H"], mu=event["mu"], V=f"{event['V']:.5f}",
                                SE=f"{event['SE']:.5f}")
            else:
                bar.set_postfix(stage=event["stage"], H=event["H"], mu=event["mu"], n=event.get("n", 0))
        manifest, nodes, _ = run_experiment(
            cfg, RUN_DIR, batch_size=BATCH_SIZE, M_validation=M_VALIDATION, progress=report,
            collect_paired=COLLECT_PAIRED
        )
    print("Готовых узлов:", int(nodes.complete.sum()))
    print("Далее: 04-estimation.ipynb с тем же RUN_ID; границы готовятся автоматически.")
elif MODE == "grid":
    nodes = run_grid(cfg, RUN_DIR)
else:
    raise ValueError("MODE должен быть fresh или grid")


Узлы:   0%|          | 0/61147 [00:00<?, ?it/s]

Готовых узлов: 61147
Далее: 04-estimation.ipynb с тем же RUN_ID; границы готовятся автоматически.


## Дальнейшая работа

В `04-estimation.ipynb` по сохранённым парным моментам готовятся границы и подбираются окна, метод оценки H и α на синтетических рынках. `05-backtest.ipynb` применяет зафиксированный выбор к реальным данным. `06-theory-vs-practice.ipynb` сравнивает стоимость и границы с аналитикой.

Для старого прогона без парной статистики используйте необязательный раздел ниже: он повторяет тест сохранённых стратегий без обучения. Новые прогоны с `COLLECT_PAIRED=True` этого не требуют.

## Парная статистика старого прогона — необязательно

По умолчанию пересчёт выключен. Для старого прогона задайте `PAIRED_RUN_ID`, включите `RECOMPUTE_PAIRED` и выполните только эту ячейку после импортов. Запуск обучения выше не нужен. Пересчёт может быть длительным; готовые узлы пропускаются при продолжении.

In [ ]:
RECOMPUTE_PAIRED = False
PAIRED_RUN_ID = "daily21_21_train20k_test20k_v1"
PAIRED_H_VALUES = None  # None — вся сетка H

if RECOMPUTE_PAIRED:
    from experiments.paired import run_paired_evaluation

    paired_run_dir = ROOT / "result_optimal_stopping" / "runs" / PAIRED_RUN_ID
    paired_output = run_paired_evaluation(paired_run_dir, H_values=PAIRED_H_VALUES)
    print("Парная статистика:", paired_output)
else:
    print("Парный пересчёт выключен; новые прогоны уже собирают парные моменты.")